In [ ]:
from notebook.services.config import ConfigManagercm = ConfigManager()cm.update('livereveal', {'width': 1920, 'height': 1080, 'scroll': True})

# Week 12: Monday, AST 5011: Astrophysical Systems

## Black Holes & AGN Feedback

### Michael Coughlin

**Reference:** Cimatti, Fraternali & Nipoti, Ch. 9

With material from Benedikt Diemer (UMD).

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom colossus.utils import constantsfrom colossus.cosmology import cosmologyimport routines as rt%matplotlib inline%config InlineBackend.figure_format = 'retina'cosmo = cosmology.setCosmology('planck18')

## Supermassive Black Holes

Nearly every massive galaxy harbors a supermassive black hole (SMBH) at its center with mass $10^6$–$10^{10}\,M_\odot$. When matter accretes onto the SMBH, the gravitational potential energy is converted to radiation:

$$L = \epsilon_{\text{rad}}\, \dot{M}\, c^2$$

where $\epsilon_{\text{rad}} \approx 0.1$ is the radiative efficiency. This is enormously efficient — about 10% of the rest-mass energy, compared to $\sim 0.7\%$ for nuclear fusion.

For an accretion rate of $1\,M_\odot/$yr: $L \approx 5.7 \times 10^{45}\,$erg/s $\sim 10^{12}\,L_\odot$.

![Accretion Schematic](figures/accretion_schematic.png)

Key observational milestones:
- Stellar/gas kinematics (HST): resolved BH sphere of influence in $\sim 100$ galaxies, establishing BH demographics
- Event Horizon Telescope (2019, 2022): direct imaging of the BH shadow in M87* ($6.5 \times 10^9\,M_\odot$) and Sgr A* ($4.1 \times 10^6\,M_\odot$)
- JWST (2023+): discovery of luminous quasars and overmassive BHs at $z > 10$, challenging seed formation models

In [ ]:
# Accretion luminosityeps_rad = 0.1L_1Msun = rt.accretionLuminosity(1.0, eps_rad)print(f'Luminosity at 1 Msun/yr accretion: {L_1Msun:.2e} erg/s')print(f'  = {L_1Msun / rt.L_sun:.2e} Lsun')

## The Eddington Limit

Radiation pressure from the accreting BH pushes outward on infalling gas. The maximum luminosity where gravity still wins is the Eddington luminosity:

$$L_{\text{Edd}} = \frac{4\pi G M_{\text{BH}} m_p c}{\sigma_T} \approx 1.3 \times 10^{38}\left(\frac{M_{\text{BH}}}{M_\odot}\right)\,\text{erg/s}$$

This sets the maximum accretion rate (the Eddington rate):

$$\dot{M}_{\text{Edd}} = \frac{L_{\text{Edd}}}{\epsilon_{\text{rad}} c^2} \approx 2.2 \times 10^{-8}\left(\frac{M_{\text{BH}}}{M_\odot}\right)\,M_\odot/\text{yr}$$

## Exercise 1: Black Hole Growth

If a BH accretes at the Eddington rate, its mass grows exponentially:

$$M_{\text{BH}}(t) = M_{\text{seed}}\, \exp\left(\frac{t}{t_{\text{Salp}}}\right)$$

where the Salpeter time $t_{\text{Salp}} = \epsilon_{\text{rad}} \times \sigma_T c / (4\pi G m_p) \approx 4.5 \times 10^7\,$yr.

Tasks:
1. Compute the Salpeter time and Eddington accretion rate for Sgr A* ($M = 4.1 \times 10^6\,M_\odot$)
2. How long does it take to grow from $100\,M_\odot$ to $10^9\,M_\odot$?
3. Compare to the age of the Universe at $z = 7$ — can we explain $10^9\,M_\odot$ quasars at $z > 7$?

In [ ]:
# Exercise 1: BH growth timescalesM_sgr = 4.1e6  # Msun (Sgr A*)# FILL IN: compute Salpeter timet_salp = ...  # FILL IN: rt.salpeterTime(eps_rad)# FILL IN: compute Eddington accretion rate for Sgr A*Mdot_edd_sgr = ...  # FILL IN: rt.eddingtonAccretionRate(M_sgr, eps_rad)print(f'Salpeter time: {t_salp:.2e} yr')print(f'Eddington accretion rate for Sgr A*: {Mdot_edd_sgr:.2e} Msun/yr')# FILL IN: time to grow from 100 to 10^9 MsunM_seed = 100.0M_final = 1e9n_efold = np.log(M_final / M_seed)t_grow = ...  # FILL IN: t_salp * n_efoldprint(f'\nE-foldings needed: {n_efold:.1f}')print(f'Time to grow 100 -> 10^9 Msun: {t_grow:.2e} yr')# Age of Universe at z=7t_z7 = cosmo.age(7.0) * 1e9print(f'Age of Universe at z=7: {t_z7:.2e} yr')print(f'Ratio t_grow / t_age(z=7): {t_grow / t_z7:.2f}')

## Demonstration: BH Growth Tracks — The Seed Mass Problem

The figure below shows $M_{\text{BH}}(z)$ for different seed masses (light: $100\,M_\odot$ remnants from Pop III stars; heavy: $10^4$–$10^5\,M_\odot$ from direct collapse) and radiative efficiencies ($\epsilon = 0.05, 0.1, 0.2$), assuming continuous Eddington-rate accretion starting at $z = 20$.

Lower $\epsilon$ allows faster growth (less energy is radiated per unit mass accreted). Light seeds ($100\,M_\odot$) struggle to reach $10^9\,M_\odot$ by $z = 7$ unless $\epsilon$ is very low, while heavy seeds ($10^5\,M_\odot$) reach it easily. This illustrates why the observed $z > 7$ quasars pose a fundamental challenge for BH formation theory.

In [ ]:
# BH growth tracks: seed mass and radiative efficiency comparison
z_growth = np.linspace(20, 0, 500)
t_growth = cosmo.age(z_growth)  # Gyr
t_form = cosmo.age(20.0)        # seed formation at z=20

seeds = [(100, 'Light: $10^2\,M_\odot$', '-'),
         (1e4, 'Heavy: $10^4\,M_\odot$', '--'),
         (1e5, 'Heavy: $10^5\,M_\odot$', ':')]

eps_vals = [0.05, 0.1, 0.2]
colors_eps = ['C0', 'C1', 'C2']

fig, ax = plt.subplots(figsize=(6, 5))
ax.set_yscale('log')
ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$M_{\rm BH}\ (M_\odot)$')
ax.set_xlim(20, 0)
ax.set_ylim(1e1, 1e11)

for M_seed, _, ls in seeds:
    for k, eps in enumerate(eps_vals):
        t_grow_yr = np.maximum((t_growth - t_form) * 1e9, 0)  # years since z=20
        M_bh = rt.bhGrowth(M_seed, t_grow_yr, eps)
        label = r'$\epsilon=%.2f$' % eps if M_seed == 100 else None
        ax.plot(z_growth, M_bh, ls=ls, color=colors_eps[k], lw=1.5, label=label)

# Mark z=7 quasar constraint
ax.axvline(7.0, ls='--', color='gray', lw=0.8)
ax.text(6.5, 2e10, '$z=7$', color='gray', fontsize=10)
ax.axhline(1e9, ls=':', color='red', lw=0.8)
ax.text(19, 1.5e9, r'$10^9\,M_\odot$ quasars', color='red', fontsize=9)

# Seed labels
for M_seed, label, ls in seeds:
    ax.plot([], [], ls=ls, color='gray', label=label)

ax.legend(fontsize=8, ncol=2, loc='lower left')
ax.set_title('BH Growth at Eddington Rate (seeded at $z=20$)')
plt.tight_layout()
plt.show()

## The $M_{\text{BH}}$–$\sigma$ Relation

One of the most remarkable discoveries in galaxy evolution is the tight correlation between SMBH mass and the velocity dispersion of the host galaxy's bulge:

$$\log_{10}\left(\frac{M_{\text{BH}}}{M_\odot}\right) = 8.49 + 4.38\,\log_{10}\left(\frac{\sigma}{200\,\text{km/s}}\right)$$

(Kormendy & Ho 2013)

This implies that the BH and its host galaxy "know about each other" despite the BH's sphere of influence being tiny ($\sim$ pc) compared to the galaxy ($\sim$ kpc). This co-evolution is mediated by AGN feedback.

## Exercise 2: The $M_{\text{BH}}$–$\sigma$ and $M_{\text{BH}}$–$M_{\text{bulge}}$ Relations1. Plot the $M_{\text{BH}}$–$\sigma$ relation for $\sigma = 50$–$400\,$km/s2. Plot the $M_{\text{BH}}$–$M_{\text{bulge}}$ relation for $M_{\text{bulge}} = 10^9$–$10^{12}\,M_\odot$3. Estimate $M_{\text{BH}}$ for the Milky Way ($\sigma \approx 105\,$km/s) and compare to Sgr A*

In [ ]:
# Exercise 2: BH scaling relationssigma_arr = np.linspace(50, 400, 100)M_bulge_arr = 10**np.linspace(9, 12, 100)# FILL IN: compute M_BH from sigma relationM_bh_sigma = ...  # FILL IN: rt.mbhSigmaRelation(sigma_arr)# FILL IN: compute M_BH from M_bulge relationM_bh_bulge = ...  # FILL IN: rt.mbhMbulgeRelation(M_bulge_arr)fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))# Left: M_BH vs sigmaax1 = axes[0]ax1.loglog(sigma_arr, M_bh_sigma, 'b-', lw=2)ax1.axvline(105, ls='--', color='orange', lw=0.8, label=r'MW ($\sigma=105$ km/s)')ax1.axhline(4.1e6, ls=':', color='gray', lw=0.8, label=r'Sgr A* ($4.1 \times 10^6 M_\odot$)')ax1.set_xlabel(r'$\sigma\ (\rm km/s)$')ax1.set_ylabel(r'$M_{\rm BH}\ (M_\odot)$')ax1.legend(fontsize=9)ax1.set_title(r'$M_{\rm BH}$–$\sigma$ relation')# Right: M_BH vs M_bulgeax2 = axes[1]ax2.loglog(M_bulge_arr, M_bh_bulge, 'r-', lw=2)ax2.set_xlabel(r'$M_{\rm bulge}\ (M_\odot)$')ax2.set_ylabel(r'$M_{\rm BH}\ (M_\odot)$')ax2.set_title(r'$M_{\rm BH}$–$M_{\rm bulge}$ relation')plt.tight_layout()plt.show()# MW predictionM_bh_MW = rt.mbhSigmaRelation(105.0)print(f'MW prediction: M_BH = {M_bh_MW:.2e} Msun')print(f'Sgr A* observed: M_BH = 4.1e+06 Msun')print(f'Ratio: {M_bh_MW / 4.1e6:.1f}')

## Demonstration: The Black Hole Sphere of Influence

The sphere of influence $r_{\text{SOI}}$ is the radius within which the BH's gravity dominates over the stellar potential:

$$r_{\text{SOI}} = \frac{G M_{\text{BH}}}{\sigma^2}$$

For the Milky Way (Sgr A*, $\sigma = 105$ km/s), $r_{\text{SOI}} \sim 2$ pc — tiny compared to the galaxy's effective radius $R_e \sim 3$ kpc. Yet the $M_{\text{BH}}$–$\sigma$ relation implies the BH and galaxy are intimately coupled despite the BH controlling only $\sim 10^{-3}$ of the galaxy's volume. This is the central puzzle of BH-galaxy co-evolution.

In [ ]:
# BH sphere of influence vs velocity dispersion
sigma_soi = np.linspace(50, 400, 100)  # km/s
M_bh_soi = rt.mbhSigmaRelation(sigma_soi)
pc_cm = 3.086e18  # 1 pc in cm

# r_SOI = G * M_BH / sigma^2
r_soi_pc = (constants.G_CGS * M_bh_soi * constants.MSUN
            / (sigma_soi * 1e5)**2 / pc_cm)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.loglog(sigma_soi, r_soi_pc, 'b-', lw=2, label=r'$r_{\rm SOI}$')

# Typical galaxy effective radius range (1-10 kpc = 1000-10000 pc)
ax.axhspan(1e3, 1e4, alpha=0.1, color='gray')
ax.text(55, 3e3, r'Typical $R_e$ (1–10 kpc)', color='gray', fontsize=9)

# Mark MW: Sgr A*, sigma = 105 km/s
sigma_MW = 105.0
r_soi_MW = (constants.G_CGS * 4.1e6 * constants.MSUN
            / (sigma_MW * 1e5)**2 / pc_cm)
ax.plot(sigma_MW, r_soi_MW, 'o', color='orange', ms=8, zorder=5)
ax.annotate(f'Sgr A* ({r_soi_MW:.1f} pc)', xy=(sigma_MW, r_soi_MW),
            xytext=(140, r_soi_MW * 0.3), fontsize=9, color='orange',
            arrowprops=dict(arrowstyle='->', color='orange'))

ax.set_xlabel(r'$\sigma\ (\rm km/s)$')
ax.set_ylabel(r'$r_{\rm SOI}\ (\rm pc)$')
ax.set_xlim(50, 400)
ax.set_ylim(0.1, 3e4)
ax.legend(fontsize=10)
ax.set_title(r'BH Sphere of Influence ($r_{\rm SOI} = GM_{\rm BH}/\sigma^2$)')
plt.tight_layout()
plt.show()

print(f'MW: r_SOI = {r_soi_MW:.1f} pc, R_e ~ 3000 pc')
print(f'r_SOI / R_e ~ {r_soi_MW / 3000:.1e}')

## AGN Feedback

AGN feedback is the process by which energy from the accreting BH couples to the surrounding gas, preventing cooling and star formation. Two main modes:

![AGN Feedback Modes](figures/agn_feedback_modes.png)

1. Quasar mode (radiative/wind): at high accretion rates ($\dot{M} \sim \dot{M}_{\text{Edd}}$), radiation drives powerful winds that can expel gas from the galaxy. This is ejective feedback — it physically removes fuel for star formation. Observed as broad absorption line (BAL) quasars and fast molecular outflows ($v \sim 1000$ km/s).

2. Radio mode (jet/maintenance): at low accretion rates, relativistic jets heat the surrounding hot gas halo, preventing it from cooling. This is preventive feedback — it maintains the quenched state. Directly observed as X-ray cavities in galaxy clusters (e.g., Perseus, MS 0735+7421), where jets have inflated bubbles that displace the intracluster medium.

The key insight: AGN feedback energy easily exceeds the binding energy of the galaxy:

$$E_{\text{AGN}} = \epsilon_{\text{rad}} M_{\text{BH}} c^2 \sim 100 \times E_{\text{bind}}$$

## Exercise 3: AGN Energy BudgetCompare the total energy released by a SMBH over its lifetime to the binding energy of the host galaxy's gas.1. Compute $E_{\text{AGN}} = \epsilon_{\text{rad}} M_{\text{BH}} c^2$ for different BH masses2. Compute the binding energy $E_{\text{bind}} \sim M_{\text{gas}} \sigma^2$ (where $M_{\text{gas}} \sim f_b M_{\text{halo}}$)3. Plot the ratio $E_{\text{AGN}} / E_{\text{bind}}$ vs halo mass

In [ ]:
# Exercise 3: AGN energy budgetM_halo_arr = 10**np.linspace(10, 15, 100)fb = cosmo.Ob0 / cosmo.Om0# Estimate BH mass from halo mass via SHMR -> bulge mass -> BH mass# Rough: M_bulge ~ 0.5 * M_star, M_star ~ 0.02 * M_halo (peak efficiency)M_star_rough = 0.02 * M_halo_arrM_bulge_rough = 0.5 * M_star_rough# FILL IN: compute M_BH from M_bulgeM_bh_arr = ...  # FILL IN: rt.mbhMbulgeRelation(M_bulge_rough)# FILL IN: AGN energy (erg)E_agn = ...  # FILL IN: eps_rad * M_bh_arr * constants.MSUN * constants.C**2# Gas binding energy: E_bind ~ M_gas * sigma^2, sigma ~ V_vir# V_vir ~ 200 * (M_halo / 10^12)^(1/3) km/sV_vir = 200.0 * (M_halo_arr / 1e12)**(1.0/3.0) * 1e5  # cm/sM_gas = fb * M_halo_arr# FILL IN: binding energy (erg)E_bind = ...  # FILL IN: M_gas * constants.MSUN * V_vir**2ratio = E_agn / E_bindplt.figure(figsize=(5, 4))plt.loglog(M_halo_arr, ratio, 'b-', lw=2)plt.axhline(1.0, ls='--', color='gray', lw=0.8, label=r'$E_{\rm AGN} = E_{\rm bind}$')plt.xlabel(r'$M_{\rm halo}\ (M_\odot)$')plt.ylabel(r'$E_{\rm AGN} / E_{\rm bind}$')plt.xlim(1e10, 1e15)plt.legend(fontsize=10)plt.title('AGN Energy vs Gas Binding Energy')plt.tight_layout()plt.show()print(f'At M_halo = 10^12: E_AGN/E_bind = {np.interp(1e12, M_halo_arr, ratio):.0f}')print(f'At M_halo = 10^14: E_AGN/E_bind = {np.interp(1e14, M_halo_arr, ratio):.0f}')

## Demonstration: The Cooling Flow Problem

In massive galaxy clusters, the hot intracluster medium (ICM) radiates X-rays via thermal bremsstrahlung. The cooling time in the cluster core can be shorter than the Hubble time, implying that gas should cool, condense, and form stars at rates of $\sim 100$–$1000\,M_\odot$/yr. This is the "cooling flow" prediction.

Observations show the opposite: star formation rates in cluster cores are typically only $\sim 1$–$10\,M_\odot$/yr — a factor of 10–100 below the cooling flow prediction. Something must be heating the gas to offset radiative cooling. The answer is radio-mode AGN feedback: jets from the central BCG's SMBH inflate cavities in the ICM, injecting $\sim 10^{44}$–$10^{46}$ erg/s of mechanical energy that balances the cooling losses.

In [ ]:
# Demonstration: Cooling flow problem — cooling luminosity vs jet power

# Typical cluster core parameters
T_keV_arr = np.array([1.0, 3.0, 5.0, 8.0])   # gas temperature (keV)
n_e_arr = np.array([0.05, 0.03, 0.02, 0.01])  # electron density (cm^-3)
r_cool_arr = np.array([50, 100, 150, 200])     # cooling radius (kpc)
cluster_names = ['Group', 'Poor cluster', 'Rich cluster', 'Massive cluster']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left panel: cooling time vs cluster temperature
ax1 = axes[0]
n_e_range = np.logspace(-3, -0.5, 100)
for T in [1.0, 3.0, 8.0]:
    t_cool = rt.coolingTime(T, n_e_range)
    ax1.loglog(n_e_range, t_cool, lw=2, label=f'$T = {T:.0f}$ keV')

ax1.axhline(cosmo.age(0.0) * 1e9, ls='--', color='gray', lw=1)
ax1.text(2e-3, cosmo.age(0.0) * 1e9 * 1.3, '$t_{\\rm Hubble}$', color='gray', fontsize=9)
ax1.axhspan(1e8, 1e9, alpha=0.1, color='blue')
ax1.text(3e-1, 3e8, 'Cooling\nflow', color='blue', fontsize=9, ha='center')
ax1.set_xlabel(r'$n_e\ (\rm cm^{-3})$')
ax1.set_ylabel(r'$t_{\rm cool}\ (\rm yr)$')
ax1.set_xlim(1e-3, 1)
ax1.set_ylim(1e6, 1e12)
ax1.legend(fontsize=9)
ax1.set_title('Cooling Time vs Density')

# Right panel: cooling luminosity vs jet power for different BH masses
ax2 = axes[1]
M_bh_jet = np.logspace(8, 10, 50)

# Cooling luminosities for representative clusters
for i, (T, n_e, r_c, name) in enumerate(zip(T_keV_arr, n_e_arr, r_cool_arr, cluster_names)):
    L_cool = rt.coolingLuminosity(T, n_e, r_c)
    ax2.axhline(L_cool, ls=':', color=f'C{i}', lw=1, alpha=0.7)
    ax2.text(1.1e8, L_cool * 1.3, name, fontsize=8, color=f'C{i}')

# Jet power at different Eddington fractions
for mdot_frac, ls in [(0.01, '-'), (0.001, '--'), (0.0001, ':')]:
    P_jet = rt.jetPower(M_bh_jet, mdot_frac, eta_jet=0.1)
    label = r'$\dot{m} = %.0e\,\dot{M}_{\rm Edd}$' % mdot_frac
    ax2.loglog(M_bh_jet, P_jet, ls=ls, color='purple', lw=2, label=label)

ax2.set_xlabel(r'$M_{\rm BH}\ (M_\odot)$')
ax2.set_ylabel(r'Power (erg/s)')
ax2.set_xlim(1e8, 1e10)
ax2.set_ylim(1e41, 1e47)
ax2.legend(fontsize=8, loc='upper left')
ax2.set_title(r'Jet Power vs $L_{\rm cool}$')

plt.tight_layout()
plt.show()

# Print a specific example: Perseus cluster
T_perseus = 6.0  # keV
n_e_perseus = 0.05  # cm^-3
r_cool_perseus = 100  # kpc
L_cool_perseus = rt.coolingLuminosity(T_perseus, n_e_perseus, r_cool_perseus)
Mdot_cool_perseus = rt.massDepositionRate(L_cool_perseus, T_perseus)
print(f'Perseus-like cluster:')
print(f'  L_cool = {L_cool_perseus:.1e} erg/s')
print(f'  Classical Mdot_cool = {Mdot_cool_perseus:.0f} Msun/yr')
print(f'  Observed SFR ~ 20-50 Msun/yr (factor ~10 suppression)')
print(f'  Jet power needed: ~{L_cool_perseus:.1e} erg/s')

## Demonstration: The $M_{\text{BH}}$–$M_{\text{halo}}$ Connection

We can connect the BH mass to the dark matter halo mass by chaining two scaling relations:

1. Via $M_{\text{bulge}}$: $M_{\text{halo}} \to M_* \to M_{\text{bulge}} \to M_{\text{BH}}$ (using approximate SHMR efficiency and bulge-to-total ratio)
2. Via $\sigma$: $M_{\text{halo}} \to V_{\text{vir}} \to \sigma \to M_{\text{BH}}$ (using the virial velocity as a proxy for $\sigma$)

Both paths give consistent results, confirming that the BH mass is ultimately set by the halo mass. The BH-to-halo mass ratio $M_{\text{BH}}/M_{\text{halo}} \sim 10^{-5}$ is remarkably small, yet AGN feedback from this tiny mass reservoir dominates galaxy evolution in massive halos.

## Demonstration: AGN Duty Cycle and the Eddington Ratio

Not all SMBHs are actively accreting. The Eddington ratio $\lambda = L_{\rm AGN}/L_{\rm Edd}$ tells us how hard a BH is working relative to its maximum. Observations show:

- Quasars at $z \sim 2$–$3$: $\lambda \sim 0.1$–$1$ (near-Eddington accretion)
- Local Seyfert galaxies: $\lambda \sim 0.01$–$0.1$
- Low-luminosity AGN (e.g., Sgr A*): $\lambda \sim 10^{-8}$ — essentially starving

The AGN duty cycle $f_{\rm duty}$ is the fraction of time a BH spends in an active phase. For quasars, $f_{\rm duty} \sim 0.01$–$0.1$ (active for $\sim 10^7$–$10^8$ yr out of the Hubble time). This has implications for BH growth: if the duty cycle is low, the BH needs to accrete at high Eddington ratios during its active phases to reach its observed mass.

In [ ]:
# AGN duty cycle and Eddington ratio demonstration

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: BH growth with intermittent accretion (duty cycle effect)
ax1 = axes[0]
t_arr = np.linspace(0, 3e9, 1000)  # 3 Gyr
M_seed_dc = 1e6  # start with 10^6 Msun

# Continuous Eddington
M_cont = rt.bhGrowth(M_seed_dc, t_arr, 0.1)

# Intermittent: duty cycle = 10%, random on/off episodes
np.random.seed(42)
n_episodes = 30
episode_len = t_arr[-1] / n_episodes
active = np.random.random(n_episodes) < 0.1  # 10% duty cycle
M_inter = np.ones_like(t_arr) * M_seed_dc
for i in range(1, len(t_arr)):
    ep = min(int(t_arr[i] / episode_len), n_episodes - 1)
    if active[ep]:
        dt = t_arr[i] - t_arr[i-1]
        M_inter[i] = rt.bhGrowth(M_inter[i-1], dt, 0.1)
    else:
        M_inter[i] = M_inter[i-1]

# Duty cycle = 50%
active50 = np.random.random(n_episodes) < 0.5
M_inter50 = np.ones_like(t_arr) * M_seed_dc
for i in range(1, len(t_arr)):
    ep = min(int(t_arr[i] / episode_len), n_episodes - 1)
    if active50[ep]:
        dt = t_arr[i] - t_arr[i-1]
        M_inter50[i] = rt.bhGrowth(M_inter50[i-1], dt, 0.1)
    else:
        M_inter50[i] = M_inter50[i-1]

ax1.semilogy(t_arr / 1e9, M_cont, 'b-', lw=2, label=r'$f_{\rm duty} = 1$ (continuous)')
ax1.semilogy(t_arr / 1e9, M_inter50, 'g--', lw=2, label=r'$f_{\rm duty} = 0.5$')
ax1.semilogy(t_arr / 1e9, M_inter, 'r:', lw=2, label=r'$f_{\rm duty} = 0.1$')
ax1.set_xlabel('Time (Gyr)')
ax1.set_ylabel(r'$M_{\rm BH}\ (M_\odot)$')
ax1.set_title('BH Growth with Intermittent Accretion')
ax1.legend(fontsize=9)

# Right: AGN luminosity function schematic — where different objects fall
ax2 = axes[1]
lambda_arr = np.logspace(-10, 0, 200)

# Schematic Eddington ratio distribution (broken power law)
phi = np.where(lambda_arr < 0.01,
               1e-2 * (lambda_arr / 0.01)**(-0.6),
               1e-2 * (lambda_arr / 0.01)**(-2.0))

ax2.loglog(lambda_arr, phi, 'k-', lw=2)
ax2.fill_between(lambda_arr, phi, alpha=0.1, color='blue')

# Annotate different AGN populations
ax2.axvspan(0.1, 1.0, alpha=0.15, color='red')
ax2.text(0.3, 5e-4, 'Quasars', fontsize=10, color='red', ha='center', rotation=90)

ax2.axvspan(0.01, 0.1, alpha=0.15, color='orange')
ax2.text(0.03, 5e-4, 'Seyferts', fontsize=10, color='orange', ha='center', rotation=90)

ax2.axvspan(1e-5, 0.01, alpha=0.1, color='green')
ax2.text(3e-4, 5e-4, 'LINERs', fontsize=10, color='green', ha='center', rotation=90)

ax2.annotate(r'Sgr A* ($\lambda \sim 10^{-8}$)', xy=(1e-8, 0.3),
             fontsize=9, color='purple',
             arrowprops=dict(arrowstyle='->', color='purple'),
             xytext=(1e-6, 2))

ax2.set_xlabel(r'Eddington ratio $\lambda = L/L_{\rm Edd}$')
ax2.set_ylabel(r'$\phi(\lambda)$ (schematic)')
ax2.set_xlim(1e-10, 2)
ax2.set_ylim(1e-5, 10)
ax2.set_title('Eddington Ratio Distribution (Schematic)')

plt.tight_layout()
plt.show()

# Sgr A* Eddington ratio
L_sgr = 1e36  # erg/s (observed X-ray luminosity of Sgr A*)
lambda_sgr = rt.eddingtonRatio(L_sgr, 4.1e6)
print(f'Sgr A*: L ~ 10^36 erg/s, L_Edd = {rt.eddingtonLuminosity(4.1e6):.1e} erg/s')
print(f'  Eddington ratio: {lambda_sgr:.1e}')

In [ ]:
# M_BH - M_halo connection via two independent scaling relation paths
M_halo_bh = 10**np.linspace(10, 15, 100)

# Path 1: M_halo -> M_star (peak efficiency ~2%) -> M_bulge (B/T~0.5) -> M_BH
M_star_est = 0.02 * M_halo_bh
M_bulge_est = 0.5 * M_star_est
M_bh_bulge_path = rt.mbhMbulgeRelation(M_bulge_est)

# Path 2: M_halo -> V_vir -> sigma (~V_vir/sqrt(2)) -> M_BH
V_vir_est = 200.0 * (M_halo_bh / 1e12)**(1.0 / 3.0)  # km/s
sigma_est = V_vir_est / np.sqrt(2)
M_bh_sigma_path = rt.mbhSigmaRelation(sigma_est)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: M_BH vs M_halo
ax1 = axes[0]
ax1.loglog(M_halo_bh, M_bh_bulge_path, 'b-', lw=2, label=r'Via $M_{\rm bulge}$')
ax1.loglog(M_halo_bh, M_bh_sigma_path, 'r--', lw=2, label=r'Via $\sigma$')
ax1.set_xlabel(r'$M_{\rm halo}\ (M_\odot)$')
ax1.set_ylabel(r'$M_{\rm BH}\ (M_\odot)$')
ax1.axvline(1e12, ls=':', color='gray', lw=0.8)
ax1.text(1.2e12, 3e5, 'MW mass', color='gray', fontsize=9)
ax1.legend(fontsize=9)
ax1.set_title(r'$M_{\rm BH}$–$M_{\rm halo}$ Connection')

# Right: M_BH/M_halo ratio
ax2 = axes[1]
ax2.loglog(M_halo_bh, M_bh_bulge_path / M_halo_bh, 'b-', lw=2, label=r'Via $M_{\rm bulge}$')
ax2.loglog(M_halo_bh, M_bh_sigma_path / M_halo_bh, 'r--', lw=2, label=r'Via $\sigma$')
ax2.set_xlabel(r'$M_{\rm halo}\ (M_\odot)$')
ax2.set_ylabel(r'$M_{\rm BH} / M_{\rm halo}$')
ax2.legend(fontsize=9)
ax2.set_title(r'BH-to-Halo Mass Ratio')

plt.tight_layout()
plt.show()

## Summary

1. SMBHs grow by accretion, with luminosity up to $L_{\text{Edd}} \propto M_{\text{BH}}$. The Salpeter time ($4.5 \times 10^7$ yr) sets the e-folding timescale for growth.

2. BH scaling relations ($M_{\text{BH}}$–$\sigma$, $M_{\text{BH}}$–$M_{\text{bulge}}$) reveal tight co-evolution between SMBHs and their host galaxies.

3. AGN feedback operates in two modes — quasar (ejective) and radio (preventive) — and releases $\sim 100\times$ more energy than the gas binding energy.

4. The cooling flow problem in clusters is resolved by radio-mode feedback: jet power from the BCG's SMBH balances radiative cooling of the ICM.

5. The AGN duty cycle ($f_{\rm duty} \sim 0.01$–$0.1$) means BHs spend most of their time quiescent, with dramatic implications for the BH growth problem at $z > 7$.

6. The BH growth problem at high redshift requires either massive seeds, super-Eddington accretion, high duty cycles, or very early formation.